# 🚗 Car Classifier Training
Training model klasifikasi merek & model mobil dari dataset Mendeley.

**Urutan:**
1. Upload dataset zip ke Google Drive
2. Mount Drive
3. Jalankan semua cell
4. Download `car_model.h5` dan `class_names.json`

In [ ]:
# Cell 1 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 - Install dependencies
!pip install tensorflow pillow matplotlib scikit-learn -q

In [ ]:
# Cell 3 - Extract dataset
# Ganti path sesuai lokasi file zip di Google Drive kamu
import zipfile, os

DATASET_ZIP = '/content/drive/MyDrive/hj3vvx5946.zip'  # <-- ganti sesuai path kamu
EXTRACT_DIR = '/content/dataset'

if not os.path.exists(EXTRACT_DIR):
    print('Extracting dataset...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Done!')

# Cari root folder dataset (folder yang berisi subfolder kelas)
for root, dirs, files in os.walk(EXTRACT_DIR):
    if len(dirs) >= 10:  # folder dengan banyak subfolder = root dataset
        DATASET_DIR = root
        print(f'Dataset root: {DATASET_DIR}')
        print(f'Jumlah kelas: {len(dirs)}')
        print(f'Kelas: {sorted(dirs)[:10]}...')
        break

In [ ]:
# Cell 4 - Load dataset & preprocessing
import tensorflow as tf
import numpy as np
import json

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f'Jumlah kelas: {NUM_CLASSES}')
print(f'Kelas: {class_names}')

# Simpan class_names
with open('/content/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
print('class_names.json tersimpan!')

In [ ]:
# Cell 5 - Augmentasi & optimasi pipeline
AUTOTUNE = tf.data.AUTOTUNE

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1),
])

def prepare(ds, augment=False):
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(AUTOTUNE)

train_ds = prepare(train_ds, augment=True)
val_ds   = prepare(val_ds,   augment=False)
print('Pipeline siap!')

In [ ]:
# Cell 6 - Bangun model (EfficientNetB0 transfer learning)
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # freeze dulu

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.4)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Cell 7 - Training fase 1 (feature extraction, 15 epoch)
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

In [ ]:
# Cell 8 - Fine-tuning (unfreeze layer atas, 10 epoch)
base_model.trainable = True

# Freeze kecuali 30 layer terakhir
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR lebih kecil
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
# Cell 9 - Evaluasi & simpan model
loss, acc = model.evaluate(val_ds)
print(f'\nValidation Accuracy: {acc*100:.2f}%')

# Simpan sebagai .h5
model.save('/content/car_model.h5')
print('car_model.h5 tersimpan!')

In [ ]:
# Cell 10 - Download hasil
from google.colab import files
files.download('/content/car_model.h5')
files.download('/content/class_names.json')
print('Download selesai! Taruh kedua file di folder models/ FastAPI kamu.')